# Rosenstein LLE — Single-window PPS surrogate rank test

Development experiment on one deterministic 60-s Processed PPG window from Session 1. The null ensemble contains exactly `M = 39` PPS surrogates generated by `src/surrogates/pps.py`. Rosenstein LLE is the scalar test statistic, and the two-sided finite-sample rank test is implemented locally in this notebook; `statistic_test.py` is not used or modified.


In [ ]:
# Cell 1 — Setup + frozen experiment configuration
import sys
import time
from importlib import import_module
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start):
    """Find the repository root from any notebook working directory."""
    for path in (start, *start.parents):
        if (path / "phase1" / "src").is_dir():
            return path
    raise FileNotFoundError("Repository root was not found.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

loader = import_module("phase1.src.dataloader.loader")
pps = import_module("phase1.src.surrogates.pps")
lyapunov = import_module("phase1.src.chaos.lyapunov")
load_segmented_session = loader.load_segmented_session
generate_pps_signal = pps.generate_pps_signal
compute_rosenstein_lle = lyapunov.compute_rosenstein_lle

SESSION = 1
SESSION_FILE = f"sample_{SESSION}.csv"
REPRESENTATION = "processed"
WINDOW_SIZE_S = 60
M = 39
ALPHA = 0.05
ALTERNATIVE = "two-sided"
MASTER_SEED = 20260828
MIN_INITIAL_PAIRS = 50
MIN_FIT_PAIRS = 30
MIN_R2 = 0.90

SEGMENTED_DATA_DIR = REPO_ROOT / "phase1" / "segmentated_data" / "dhdata"
RHO_PATH = (REPO_ROOT / "phase1" / "results" / "pps" /
            "pps_radius_calibration" / "csv" / "pps_radius_session_01.csv")
LLE_PATH = (REPO_ROOT / "phase1" / "results" / "lle" / "processed" /
            "session_01_processed_rosenstein_lle.csv")
STATE_NAMES = {0: "Awake", 1: "Drowsy"}

assert M == 39
assert ALTERNATIVE == "two-sided"
assert np.isclose(2 / (M + 1), ALPHA)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
print(f"Session={SESSION}, representation={REPRESENTATION}, window={WINDOW_SIZE_S} s")
print(f"M={M}, alternative={ALTERNATIVE}, alpha={ALPHA}, seed={MASTER_SEED}")


In [ ]:
# Cell 2 — Deterministically select one window and match rho/configuration
batches, segmented_metadata = load_segmented_session(
    SESSION_FILE,
    data_dir=SEGMENTED_DATA_DIR,
    window_sizes=WINDOW_SIZE_S,
    representation=REPRESENTATION,
    stationarity_only=True,
)
batch = batches[WINDOW_SIZE_S]

# Frozen selection rule: minimum window_id among eligible Processed windows.
selected_index = int(np.argmin(np.asarray(batch["window_id"], dtype=int)))
WINDOW_ID = int(batch["window_id"][selected_index])
STATE = STATE_NAMES[int(batch["label"][selected_index])]
FS = float(batch["fs"])
signal = np.asarray(batch["signal"][selected_index], dtype=float)

rho_table = pd.read_csv(RHO_PATH)
rho_match = rho_table.loc[
    rho_table["session"].eq(SESSION)
    & rho_table["representation"].str.casefold().eq(REPRESENTATION.casefold())
    & rho_table["state"].str.casefold().eq(STATE.casefold())
    & rho_table["window_size"].eq(WINDOW_SIZE_S)
    & rho_table["window_id"].eq(WINDOW_ID)
]
assert len(rho_match) == 1, "Expected exactly one matching PPS rho row."
rho_row = rho_match.iloc[0]
RHO_STAR = float(rho_row["rho_star"])

# Reuse the exact frozen estimator configuration already attached to this window.
lle_table = pd.read_csv(LLE_PATH)
lle_match = lle_table.loc[
    lle_table["session"].eq(SESSION)
    & lle_table["representation"].str.casefold().eq(REPRESENTATION.casefold())
    & lle_table["state"].str.casefold().eq(STATE.casefold())
    & lle_table["window_size_s"].eq(WINDOW_SIZE_S)
    & lle_table["window_id"].eq(WINDOW_ID)
]
assert len(lle_match) == 1, "Expected exactly one matching LLE result row."
lle_reference = lle_match.iloc[0]

assert signal.ndim == 1 and signal.size == round(WINDOW_SIZE_S * FS)
assert np.all(np.isfinite(signal)) and np.std(signal) > 0
assert np.isclose(float(rho_row["sampling_rate"]), FS, atol=1e-6)
assert int(rho_row["N"]) == signal.size
assert bool(lle_reference["analysis_included"])
assert bool(lle_reference["valid"])

selection_summary = pd.DataFrame([{
    "session": SESSION, "representation": REPRESENTATION,
    "state": STATE, "window_size_s": WINDOW_SIZE_S,
    "window_id": WINDOW_ID, "sampling_rate_hz": FS,
    "n_samples": signal.size, "rho_star": RHO_STAR,
    "selection_rule": "minimum eligible window_id",
}])
display(selection_summary.round(6))


In [ ]:
# Cell 3 — Compute and verify the original Rosenstein LLE
LLE_CONFIG = {
    "sampling_rate": FS,
    "m": int(lle_reference["m"]),
    "tau_samples": int(lle_reference["tau_samples"]),
    "fit_start_s": float(lle_reference["fit_start_s"]),
    "fit_end_s": float(lle_reference["fit_end_s"]),
    "max_follow_s": float(lle_reference["max_follow_s"]),
    # Freeze the original-window Theiler period for every surrogate.
    "theiler_s": float(lle_reference["theiler_s"]),
    "min_initial_pairs": MIN_INITIAL_PAIRS,
    "min_fit_pairs": MIN_FIT_PAIRS,
    "min_r2": MIN_R2,
}
original_lle = compute_rosenstein_lle(signal, **LLE_CONFIG)
assert original_lle.valid, original_lle.qc_reason
assert np.isclose(
    original_lle.lle, float(lle_reference["lle_1_per_s"]),
    rtol=0.0, atol=1e-12,
), "Recomputed original LLE does not match its saved result."

original_summary = pd.DataFrame([{
    "LLE_1_per_s": original_lle.lle, "fit_R2": original_lle.fit_r2,
    "fit_start_s": original_lle.fit_start_s, "fit_end_s": original_lle.fit_end_s,
    "m": LLE_CONFIG["m"], "tau_samples": LLE_CONFIG["tau_samples"],
    "theiler_samples": original_lle.theiler_samples, "theiler_s": original_lle.theiler_s,
    "n_pairs_initial": original_lle.n_pairs_initial,
    "n_pairs_fit_min": original_lle.n_pairs_fit_min,
    "valid": original_lle.valid, "qc_reason": original_lle.qc_reason,
}])
display(original_summary.round(6))


In [ ]:
# Cell 4 — Generate exactly 39 PPS surrogates and compute their LLE statistics
seed_sequences = np.random.SeedSequence(MASTER_SEED).spawn(M)
surrogate_seeds = [
    int(child.generate_state(1, dtype=np.uint64)[0]) for child in seed_sequences
]
surrogate_signals = np.empty((M, signal.size), dtype=float)
surrogate_lle_results = []
surrogate_rows = []
started = time.perf_counter()

for surrogate_id, seed in enumerate(surrogate_seeds, start=1):
    surrogate = generate_pps_signal(
        signal=signal, tau=LLE_CONFIG["tau_samples"],
        m=LLE_CONFIG["m"], rho=RHO_STAR,
        rng=np.random.default_rng(seed), return_indices=False,
    )
    result = compute_rosenstein_lle(surrogate, **LLE_CONFIG)
    surrogate_signals[surrogate_id - 1] = surrogate
    surrogate_lle_results.append(result)
    surrogate_rows.append({
        "surrogate_id": surrogate_id, "seed": seed,
        "lle_1_per_s": result.lle, "fit_r2": result.fit_r2,
        "theiler_samples": result.theiler_samples,
        "n_pairs_initial": result.n_pairs_initial,
        "n_pairs_fit_min": result.n_pairs_fit_min,
        "valid": result.valid, "qc_reason": result.qc_reason,
    })
    if surrogate_id % 10 == 0:
        print(f"Completed {surrogate_id}/{M} surrogates")

elapsed_s = time.perf_counter() - started
surrogate_table = pd.DataFrame(surrogate_rows)
assert surrogate_signals.shape == (M, signal.size)
assert len(surrogate_table) == M and len(set(surrogate_seeds)) == M
assert np.all(np.isfinite(surrogate_signals))
assert np.all(np.isfinite(surrogate_table["lle_1_per_s"]))
assert (surrogate_table["theiler_samples"] == original_lle.theiler_samples).all()

print(f"Generated and evaluated {M} surrogates in {elapsed_s:.2f} s")
display(surrogate_table.round(6))
display(
    surrogate_table.groupby(["valid", "qc_reason"], as_index=False)
    .size().rename(columns={"size": "n_surrogates"})
)


In [ ]:
# Cell 5 — Local finite-sample two-sided surrogate rank test
def two_sided_surrogate_rank_test(original_value, surrogate_values, alpha=0.05):
    """Exact Monte Carlo rank test with +1 correction and inclusive ties."""
    original_value = float(original_value)
    values = np.asarray(surrogate_values, dtype=float)
    if values.ndim != 1 or values.size == 0 or not np.all(np.isfinite(values)):
        raise ValueError("surrogate_values must be a finite, non-empty 1D array.")
    n_lower = int(np.count_nonzero(values <= original_value))
    n_upper = int(np.count_nonzero(values >= original_value))
    n_equal = int(np.count_nonzero(values == original_value))
    n_less = int(np.count_nonzero(values < original_value))
    denominator = values.size + 1
    p_lower = (n_lower + 1) / denominator
    p_upper = (n_upper + 1) / denominator
    p_value = min(1.0, 2.0 * min(p_lower, p_upper))
    reject = bool(p_value <= alpha)
    direction = "none"
    if reject and p_upper < p_lower:
        direction = "higher"
    elif reject and p_lower < p_upper:
        direction = "lower"
    return {
        "M": int(values.size),
        "rank_ascending": float(1 + n_less + 0.5 * n_equal),
        "n_surrogates_le_original": n_lower,
        "n_surrogates_ge_original": n_upper,
        "p_lower": float(p_lower), "p_upper": float(p_upper),
        "p_two_sided": float(p_value), "alpha": float(alpha),
        "reject_H0": reject, "direction_if_rejected": direction,
    }


# Keep the prespecified M=39 finite slopes; report QC without post-hoc removal.
surrogate_lle_values = surrogate_table["lle_1_per_s"].to_numpy(dtype=float)
rank_result = two_sided_surrogate_rank_test(
    original_lle.lle, surrogate_lle_values, alpha=ALPHA
)
assert rank_result["M"] == M
assert np.isclose(rank_result["p_two_sided"], 2 / (M + 1))

valid_values = surrogate_table.loc[
    surrogate_table["valid"], "lle_1_per_s"
].to_numpy(dtype=float)
qc_sensitivity = two_sided_surrogate_rank_test(
    original_lle.lle, valid_values, alpha=ALPHA
)
test_summary = pd.DataFrame([{
    "original_lle_1_per_s": original_lle.lle,
    "surrogate_mean": np.mean(surrogate_lle_values),
    "surrogate_median": np.median(surrogate_lle_values),
    "surrogate_sd": np.std(surrogate_lle_values, ddof=1),
    "surrogate_min": np.min(surrogate_lle_values),
    "surrogate_max": np.max(surrogate_lle_values),
    **rank_result,
}])
display(test_summary.round(6))
display(pd.DataFrame([{
    "analysis": "QC sensitivity: valid surrogate fits only",
    "M_retained": qc_sensitivity["M"],
    "p_two_sided": qc_sensitivity["p_two_sided"],
    "reject_H0": qc_sensitivity["reject_H0"],
}]).round(6))
print("H0: original-window LLE is exchangeable with PPS-surrogate LLE values.")
print("With M=39, the minimum attainable two-sided p-value is 2/(M+1)=0.05.")


In [ ]:
# Cell 6 — Signal, rank, and divergence diagnostics
preview_n = min(signal.size, round(10 * FS))
preview_time = np.arange(preview_n) / FS
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(preview_time, signal[:preview_n], color="black", lw=1.2, label="Original")
for index, color in zip(range(3), ("#0072B2", "#D55E00", "#009E73")):
    axes[0].plot(
        preview_time, surrogate_signals[index, :preview_n],
        color=color, alpha=0.65, lw=0.8, label=f"PPS {index + 1}",
    )
axes[0].set(title="10-s signal sanity check", xlabel="Time [s]", ylabel="PPG amplitude")
axes[0].legend(frameon=False, ncol=2)

colors = np.where(surrogate_table["valid"], "#0072B2", "#D55E00")
axes[1].scatter(surrogate_lle_values, np.zeros(M), c=colors, s=34, alpha=0.8, label="PPS LLE")
axes[1].axvline(original_lle.lle, color="black", lw=2, label="Original LLE")
axes[1].set(title="Observed rank within PPS null ensemble", xlabel="LLE [1/s]", yticks=[])
axes[1].legend(frameon=False)
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
for index, result in enumerate(surrogate_lle_results):
    ax.plot(
        result.time_s, result.mean_log_distance,
        color="#0072B2" if result.valid else "#D55E00",
        alpha=0.12, lw=0.8, label="PPS curves" if index == 0 else None,
    )
ax.plot(
    original_lle.time_s, original_lle.mean_log_distance,
    color="black", lw=2.0, label="Original",
)
ax.axvspan(
    LLE_CONFIG["fit_start_s"], LLE_CONFIG["fit_end_s"],
    color="grey", alpha=0.16, label="Frozen fit region",
)
ax.set(
    title="Rosenstein divergence curves: original vs 39 PPS surrogates",
    xlabel="Follow time [s]", ylabel="Mean log distance",
)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


# Single-window PPS surrogate test — Report

## Configuration

- Window: Session 1, Processed, Awake, 60 s, `window_id=1` (minimum eligible ID)
- Sampling rate: 50 Hz; samples: 3,000
- PPS core: `generate_pps_signal()`; calibrated `rho_star=18.184066`
- Null ensemble: `M=39`, fixed seed `20260828`
- LLE: `m=8`, `tau=8` samples, frozen Theiler window 28 samples, fit 0.80–1.30 s
- Test: finite-sample two-sided rank test with the `+1` correction; `alpha=0.05`

## Result

- Original LLE: **0.91620 1/s** (`R²=0.98953`, QC `ok`)
- PPS LLE: median **0.64230 1/s**, mean **0.64103 1/s**, range **0.54915–0.74500 1/s**
- Original ascending rank among the 40 values: **40/40**
- Surrogates `<=` original: **39/39**; surrogates `>=` original: **0/39**
- Exact two-sided `p = 0.050`; decision rule `p <= 0.05` gives a boundary rejection in the **higher** direction.

## QC and interpretation

Six surrogate fits have `R² < 0.90`, although all 39 LLE slopes are finite and have sufficient pair support. They remain in the prespecified `M=39` primary rank ensemble. As a sensitivity description only, retaining the 33 QC-valid surrogate fits gives `p=2/34=0.0588`, which does not cross 0.05.

This window's observed LLE is therefore more extreme than every generated PPS surrogate, but the result sits exactly at the resolution limit of a two-sided test with 39 surrogates and is sensitive to LLE fit QC. It is a single-window development result—not session- or population-level evidence—and a positive LLE alone is not proof of deterministic chaos.
